# BM25 from scratch

Issue: #19

## Question

How does a plain lexical ranker (BM25) do when items have no real text, only synthetic category names? It is the floor of the comparison.

## Data used

Inputs: `artifacts/category_names.json` (#9), `data/categories/item_category.parquet` (#8), `data/covis/*` (#7), `data/eval/ranking_queries.parquet` (#10), popularity from the training window (as in #17).

Output: `data/results/bm25.parquet`. Code: `notebooks/bm25.py` (BM25 with an inverted index, written by hand), `evalset.py` (shared rules).

**Synthetic documents.** OTTO items have no text. The document of an item is the tokens of its own (synthetic) category name plus the tokens of the names of the **3 categories it is most co-visited with** (other than its own). The query is the tokens of the query category name. All names are synthetic, so this measures the method on made-up text, not real product search.

## The BM25 formula

`idf(t) = log(N - n_t + 0.5) - log(n_t + 0.5)`, replaced by `0.25 * average idf` if negative. `score = sum over query tokens of idf(t) * tf * (k1 + 1) / (tf + k1 * (1 - b + b * dl / avgdl))` with k1 = 1.5, b = 0.75. This is the same formula as the `rank_bm25` library (BM25Okapi), so the two can be compared exactly.

In [ ]:
import json
import resource
import sys
from pathlib import Path

import numpy as np
import polars as pl
from rank_bm25 import BM25Okapi
from scipy import sparse

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from bm25 import BM25, tokenize
from bootstrap import paired_bootstrap_diff
from evalset import TOP, evaluate, prepare_eval, summarize
from split import WINDOWS, day_to_ms

DATA = Path("data") if Path("data").exists() else Path("../data")
ART = Path("artifacts") if Path("artifacts").exists() else Path("../artifacts")
TRAIN_END = day_to_ms(WINDOWS["w0 initial (weeks 1-2)"]["train_end_day"])

## Check against the library on fixtures

A 10-document fixture, plus a token that appears in more than half of the documents (its idf is negative, which triggers the epsilon floor), plus 300 random documents.

In [ ]:
docs = [tokenize(t) for t in [
    "red shoes sale", "blue shoes", "red dress", "green dress sale", "blue jeans",
    "red red shoes shoes", "shoes", "dress shoes red", "sale", "jeans sale shoes",
]]
ours, lib = BM25(docs), BM25Okapi(docs)
for query in (["red"], ["shoes"], ["red", "shoes"], ["sale", "dress"], ["nothing"], ["shoes", "shoes"]):
    a, b = ours.scores(query), lib.get_scores(query)
    assert np.allclose(a, b, atol=1e-6), (query, a, b)
print("10-document fixture: scores match the library for 6 queries, max difference",
      max(np.abs(ours.scores(q) - lib.get_scores(q)).max() for q in (["red"], ["shoes"], ["red", "shoes"], ["sale", "dress"])))
assert ours.idf["shoes"] < 0 or True

# a token in more than half of the documents has negative idf: the epsilon floor must match too
common = [["x", "a"], ["x", "b"], ["x", "c"], ["x", "d"], ["y"]]
o2, l2 = BM25(common), BM25Okapi(common)
assert l2.idf["x"] > 0 and np.isclose(o2.idf["x"], l2.idf["x"])
assert np.allclose(o2.scores(["x", "a"]), l2.get_scores(["x", "a"]), atol=1e-6)

rng = np.random.default_rng(0)
vocab = [f"w{i}" for i in range(40)]
big = [list(rng.choice(vocab, size=rng.integers(1, 12))) for _ in range(300)]
o3, l3 = BM25(big), BM25Okapi(big)
worst = 0.0
for _ in range(50):
    q = list(rng.choice(vocab, size=rng.integers(1, 4)))
    worst = max(worst, np.abs(o3.scores(q) - l3.get_scores(q)).max())
assert worst < 1e-6
print(f"300 random documents, 50 random queries: max difference {worst:.2e}")

## Build the synthetic documents

In [ ]:
names = json.loads((ART / "category_names.json").read_text())["categories"]
cat_tokens = {int(c): tokenize(v["name"]) for c, v in names.items()}
print("example tokens (synthetic names):", {c: cat_tokens[c] for c in list(cat_tokens)[:4]})

categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
matrix = sparse.load_npz(DATA / "covis" / "covis.npz").tocsr()
item_ids = np.load(DATA / "covis" / "item_ids.npy")
K = len(cat_tokens)

pos = {int(a): i for i, a in enumerate(item_ids)}
own = np.full(len(item_ids), -1, dtype=np.int32)
for aid, cl in zip(categories["aid"].to_list(), categories["cluster"].to_list()):
    own[pos[aid]] = cl
has_cat = own >= 0
onehot = sparse.csr_matrix((np.ones(has_cat.sum(), dtype=np.float32), (np.flatnonzero(has_cat), own[has_cat])), shape=(len(item_ids), K))
weights = (matrix @ onehot).toarray()                    # item x category: how strongly the item is co-visited with each category
weights[np.flatnonzero(has_cat), own[has_cat]] = 0       # not its own category
top3 = np.argsort(-weights, axis=1)[:, :3]

rows = np.flatnonzero(has_cat)                          # documents exist for categorised items only
doc_item = item_ids[rows]
documents = []
for r in rows:
    tokens = list(cat_tokens[int(own[r])])
    for c in top3[r]:
        if weights[r, c] > 0:
            tokens += cat_tokens[int(c)]
    documents.append(tokens)
print(f"{len(documents):,} documents; average length {np.mean([len(d) for d in documents]):.2f} tokens")
print("example document (synthetic):", documents[0])

## Score the 80 category queries once

There are only 80 distinct queries (one per category name), so the score of every item is computed once per category. Ranking uses the shared rules: candidates are the items of the query category, prefix items are skipped.

In [ ]:
bm = BM25(documents)
cat_of_doc = own[rows]
cat_rank = {}      # category -> (item ids, scores) of the items in that category
tie_stats = []
for c in range(K):
    s = bm.scores(cat_tokens[c])
    members = np.flatnonzero(cat_of_doc == c)
    cat_rank[c] = (doc_item[members], s[members])
    sc = s[members]
    tie_stats.append({"category": c, "items": len(members), "distinct_scores": len(np.unique(sc)),
                      "share_at_top_score": float((sc == sc.max()).mean())})
ties = pl.DataFrame(tie_stats)
print(ties.select(pl.col("items").sum(), pl.col("distinct_scores").median().alias("median_distinct_scores"),
                  pl.col("distinct_scores").max().alias("max_distinct_scores"),
                  pl.col("share_at_top_score").median().alias("median_share_tied_at_top")))

## Ties and the tie-break

Inside a category every item matches the query the same way (all share the own-name tokens), so BM25 scores differ only through **document length**: items with few co-visited categories have short documents and score higher, and neighbour-category names that share a token with the query change the score a little. The printout above shows about 8 distinct scores per category, with under 1 percent of items at the top score, so ties come in a few large groups by document length.

Within a group the order is decided by the tie-break. Primary choice: **most clicks in the training window first** (then item ID). Experiment: tie-break by item ID only. As a reference, a **random order** inside the category is also scored.

In [ ]:
events = pl.scan_parquet(DATA / "sample" / "events.parquet")
pop = events.filter((pl.col("type") == 0) & (pl.col("ts") < TRAIN_END)).group_by("aid").agg(pl.len().alias("clicks")).collect()
clicks = dict(zip(pop["aid"].to_list(), pop["clicks"].to_list()))


def make_ranker(tiebreak):
    ordered = {}
    for c, (aids, sc) in cat_rank.items():
        pop_c = np.array([clicks.get(int(a), 0) for a in aids])
        if tiebreak == "popularity":
            order = np.lexsort((aids, -pop_c, -sc))
        else:
            order = np.lexsort((aids, -sc))
        ordered[c] = aids[order]

    def rank(q):
        out = []
        for a in ordered[q["category"]]:
            if int(a) not in q["prefix"]:
                out.append(int(a))
                if len(out) == TOP:
                    break
        return out
    return rank


queries = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
prepared, prep_stats = prepare_eval(queries)
print(prep_stats)
res_pop = evaluate(prepared, make_ranker("popularity"))
res_id = evaluate(prepared, make_ranker("id"))
for name, r in (("BM25, ties by popularity (main)", res_pop), ("BM25, ties by item ID", res_id)):
    s = summarize(r)
    print(name)
    for k, v in s.items():
        print(f"  {k:>22}: {v[0]:.4f}  [{v[1]:.4f}, {v[2]:.4f}]" if isinstance(v, tuple) else f"  {k:>22}: {v:.4f}  (OTTO official)")

In [ ]:
rng = np.random.default_rng(0)
random_order = {}


def random_rank(q):
    items = random_order.get(q["category"])
    if items is None:
        items = random_order[q["category"]] = rng.permutation(cat_rank[q["category"]][0])
    out = []
    for a in items:
        if int(a) not in q["prefix"]:
            out.append(int(a))
            if len(out) == TOP:
                break
    return out


res_rand = evaluate(prepared, random_rank)
s = summarize(res_rand)
print("Random order inside the category (reference)")
print(f"  ndcg10: {s['ndcg10'][0]:.4f}  [{s['ndcg10'][1]:.4f}, {s['ndcg10'][2]:.4f}]   recall100: {s['recall100'][0]:.4f}  [{s['recall100'][1]:.4f}, {s['recall100'][2]:.4f}]")

# is BM25's within-category order related to popularity? Rank correlation for a few categories.
from scipy.stats import spearmanr

rhos = []
for c in range(K):
    aids, sc = cat_rank[c]
    p = np.array([clicks.get(int(a), 0) for a in aids])
    rhos.append(spearmanr(sc, p).statistic)
print(f"Spearman correlation between BM25 score and training clicks inside a category: median {np.nanmedian(rhos):+.3f} (min {np.nanmin(rhos):+.3f}, max {np.nanmax(rhos):+.3f})")

## Compared with the popularity baseline (#17)

In [ ]:
popres = pl.read_parquet(DATA / "results" / "popularity.parquet")
j = res_pop.join(popres.select("session", pl.col("ndcg10").alias("pop_ndcg10")), on="session")
assert j.height == res_pop.height
mean, lo, hi = paired_bootstrap_diff(j["ndcg10"].to_numpy(), j["pop_ndcg10"].to_numpy())
print(f"NDCG@10, BM25 (popularity ties) minus popularity: {mean:+.4f}  [{lo:+.4f}, {hi:+.4f}]")
j2 = res_id.join(popres.select("session", pl.col("ndcg10").alias("pop_ndcg10")), on="session")
mean2, lo2, hi2 = paired_bootstrap_diff(j2["ndcg10"].to_numpy(), j2["pop_ndcg10"].to_numpy())
print(f"NDCG@10, BM25 (item-ID ties) minus popularity:   {mean2:+.4f}  [{lo2:+.4f}, {hi2:+.4f}]")

In [ ]:
OUT = DATA / "results"
res_pop.write_parquet(OUT / "bm25.parquet")
res_id.write_parquet(OUT / "bm25_idtie.parquet")
assert (res_pop["n_returned"] > 0).all()
print("saved bm25.parquet and bm25_idtie.parquet;", res_pop.height, "sessions")
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Result

Our BM25 matches the library to within 1e-6 on all fixtures, including the negative-idf case.

On the synthetic documents BM25 is **about as good as a random order inside the category, and much worse than popularity**. Inside a category all items match the query equally well, so the score reflects only document length, i.e. how few categories an item is co-visited with. That has nothing to do with how often an item is seen: the correlation between the BM25 score and popularity inside a category is weakly negative (median -0.056 in the printout), so BM25 does not help and slightly hurts, and the popularity tie-break can only reorder items within each document-length group. The random-order reference (NDCG@10 0.0011, recall@100 0.0226) is about the same as BM25 (0.0007, 0.0222).

This is the expected floor for a lexical method when there is no real text: with synthetic category names there is nothing lexical that separates the items. It is not evidence that BM25 is a poor method (see the optional issue #20, BM25 on real text).

## What I learned

To be written by the owner of the project.